# Load a PDF and Inspect the Raw Output

In [30]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("chunking.pdf")
pages = loader.load()

print(f"Pages loaded: {len(pages)}")
print(f"\nFirst page content (first 500 chars):\n{pages[0].page_content[:500]}")
print(f"\nMetadata on page 0: {pages[0].metadata}")

Pages loaded: 5

First page content (first 500 chars):
RAG Chunking Demo
1
 Beispieldokument fuer RAG-Chunking
Ein kurzes, bewusst heterogenes Dokument mit klaren Abschnitten, Listen, Tabelle,
Querverweisen und einer reinen Bildseite.
Szenario
Die fiktive Firma Nordstern Logistik betreibt ein automatisiertes Lager. Das Dokument beschreibt
einen Pilotversuch zur Senkung von Fehlkommissionierungen. Es ist so aufgebaut, dass
verschiedene Chunking-Strategien direkt verglichen werden koennen.
Empfohlene Demo-Fragen
1. Was war das Hauptziel des Pilotversu

Metadata on page 0: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-16T11:45:20+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-07-16T11:45:20+00:00', 'subject': '(unspecified)', 'title': 'RAG Chunking Beispieldokument', 'trapped': '/False', 'source': 'chunking.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


# See the Image Problem First-Hand

In [34]:
# Find pages where PyPDF extracted almost no text
empty_pages = [p for p in pages if len(p.page_content.strip()) < 100]
print(f"Pages with little/no text: {len(empty_pages)}")
for p in empty_pages:
    print(f"  Page {p.metadata['page']}: '{p.page_content[:80]}'")

Pages with little/no text: 1
  Page 3: ''


# Naive Fixed-Size Chunking

In [43]:
from langchain_text_splitters import CharacterTextSplitter

splitter_naive = CharacterTextSplitter(chunk_size=200, chunk_overlap=20,  separator=" " )
naive_chunks = splitter_naive.split_documents(pages)

print(f"Chunks created: {len(naive_chunks)}")
print(f"\nChunk 0:\n{naive_chunks[0].page_content}")
print(f"\nChunk 1:\n{naive_chunks[1].page_content}")

Chunks created: 25

Chunk 0:
RAG Chunking Demo
1
 Beispieldokument fuer RAG-Chunking
Ein kurzes, bewusst heterogenes Dokument mit klaren Abschnitten, Listen, Tabelle,
Querverweisen und einer reinen Bildseite.
Szenario
Die fiktive

Chunk 1:
fiktive Firma Nordstern Logistik betreibt ein automatisiertes Lager. Das Dokument beschreibt
einen Pilotversuch zur Senkung von Fehlkommissionierungen. Es ist so aufgebaut, dass
verschiedene


# Recursive Splitting — The Better Default

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Skip the first page (page 0) — it should not end up in chroma_db
pages_for_db = [p for p in pages if p.metadata["page"] != 0]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(pages_for_db)

print(f"Chunks created: {len(chunks)}")
print(f"\nChunk 0:\n{chunks[0].page_content}")
print(f"\nChunk metadata: {chunks[0].metadata}")
print(f"\nChunk 1:\n{chunks[1].page_content}")
print(f"\nChunk metadata: {chunks[1].metadata}")

# 6. Add Custom Metadata

In [16]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i
    chunk.metadata["document_category"] = "company-policy"
    chunk.metadata["char_count"] = len(chunk.page_content)

print(chunks[5].metadata)

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-16T11:45:20+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-07-16T11:45:20+00:00', 'subject': '(unspecified)', 'title': 'RAG Chunking Beispieldokument', 'trapped': '/False', 'source': 'chunking.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'chunk_index': 5, 'document_category': 'company-policy', 'char_count': 49}


# Local Embeddings with sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Multilingual model — handles German documents and queries much better
local_embedder = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True},
)

sample_text = chunks[0].page_content
vector = local_embedder.embed_query(sample_text)

print(f"Vector dimensions: {len(vector)}")
print(f"First 10 values: {vector[:10]}")

# 8. OpenAI Embeddings for Comparison

In [47]:
from langchain_openai import OpenAIEmbeddings

openai_embedder = OpenAIEmbeddings(model="text-embedding-3-small")

vector_openai = openai_embedder.embed_query(sample_text)
print(f"OpenAI vector dimensions: {len(vector_openai)}")

OpenAI vector dimensions: 1536


# Store in Chroma with Metadata

In [ ]:
import chromadb
from langchain_community.vectorstores import Chroma

# Delete the old collection first, otherwise every run appends duplicates
client = chromadb.PersistentClient(path="./chroma_db")
try:
    client.delete_collection("company-docs")
except Exception:
    pass

# Use local embedder for the demo (no API key needed)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=local_embedder,
    persist_directory="./chroma_db",
    collection_name="company-docs",
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Documents in store: {vectorstore._collection.count()}")

#  Inspect What’s Stored

In [49]:
# Peek at what's actually in the store
collection = vectorstore._collection
results = collection.get(limit=3, include=["documents", "metadatas"])

for i in range(len(results["documents"])):
    print(f"\n--- Chunk {i} ---")
    print(f"Text: {results['documents'][i][:150]}...")
    print(f"Metadata: {results['metadatas'][i]}")


--- Chunk 0 ---
Text: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalis...
Metadata: {'producer': 'pdfTeX-1.40.25', 'chunk_index': 0, 'creationdate': '2024-04-10T21:11:43+00:00', 'moddate': '2024-04-10T21:11:43+00:00', 'page': 0, 'subject': '', 'author': '', 'creator': 'LaTeX with hyperref', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'document_category': 'company-policy', 'trapped': '/False', 'keywords': '', 'page_label': '1', 'total_pages': 15, 'title': '', 'char_count': 492, 'source': 'sample.pdf'}

--- Chunk 1 ---
Text: University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract...
Metadata: {'keywords': '', 'title': '', 'moddate': '2024-04-10T21:11:43+00:00', 'creator': 'LaTeX with hyperref', 'subject': '', 'ptex.fullbanner': '